# RAG + RAGAS — Walkthrough Interativo

Este notebook guia você pelo processo completo:
1. Montar o pipeline RAG
2. Gerar o Golden Dataset automaticamente
3. Avaliar com métricas RAGAS
4. Interpretar os resultados


In [5]:
# Setup inicial
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Adiciona src/ ao path (execute a partir da raiz do projeto)
project_root = Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

# Carrega ANTHROPIC_API_KEY do arquivo .env
load_dotenv(project_root / '.env')

if not os.getenv('ANTHROPIC_API_KEY'):
    raise EnvironmentError("ANTHROPIC_API_KEY não encontrada. Configure o arquivo .env na raiz do projeto.")

print('✅ Setup OK')

✅ Setup OK


---
## Parte 1 — Pipeline RAG

Monta o pipeline completo: carrega documentos → cria chunks → embeddings → ChromaDB.

In [6]:
from rag_pipeline import RAGPipeline

rag = RAGPipeline()
rag.build()  # processa os documentos de data/sample_docs/

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [ ]:
# Teste rápido — faça uma pergunta
resultado = rag.query('Quando foi cunhado o termo inteligência artificial?')

print(f"❓ Pergunta: {resultado['question']}")
print(f"💬 Resposta: {resultado['answer']}")
print(f"\n📄 Contextos recuperados ({len(resultado['contexts'])}):'")
for i, ctx in enumerate(resultado['contexts'], 1):
    print(f"  [{i}] {ctx[:200]}...")

---
## Parte 2 — Geração do Golden Dataset com RAGAS

O RAGAS analisa os documentos e gera **perguntas + ground truth** automaticamente.

Isso resolve o maior desafio na avaliação de RAG: criar o dataset sem trabalho manual.

```
Seus documentos
      │
      ▼
  RAGAS TestsetGenerator
      │
      ▼
  Question + Ground Truth  ← golden dataset!
```

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_anthropic import ChatAnthropic
from langchain_huggingface import HuggingFaceEmbeddings

# Carrega documentos
DOCS_DIR = project_root / 'data' / 'sample_docs'
loader = DirectoryLoader(str(DOCS_DIR), glob='**/*.txt', loader_cls=TextLoader,
                         loader_kwargs={'encoding': 'utf-8'})
docs = loader.load()
print(f'📂 {len(docs)} documento(s) carregado(s)')

In [ ]:
# Configura o gerador RAGAS (API 0.4.x)
llm = LangchainLLMWrapper(ChatAnthropic(model='claude-haiku-4-5-20251001', temperature=0.3))
embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
)

generator = TestsetGenerator(llm=llm, embedding_model=embeddings)

# Gera 8 perguntas
testset = generator.generate_with_langchain_docs(
    documents=docs,
    testset_size=8,
)

df_golden = testset.to_pandas()
print(f'✅ {len(df_golden)} perguntas geradas!')
df_golden[['user_input', 'reference']].head()

In [ ]:
# Inspeciona as perguntas geradas
# Na API ragas 0.4.x as colunas são: user_input, reference, synthesizer_name
col_question   = 'user_input'   if 'user_input'   in df_golden.columns else 'question'
col_truth      = 'reference'    if 'reference'    in df_golden.columns else 'ground_truth'
col_type       = 'synthesizer_name' if 'synthesizer_name' in df_golden.columns else 'question_type'

for tipo in df_golden[col_type].unique():
    row = df_golden[df_golden[col_type] == tipo].iloc[0]
    print(f"\n🏷️  Tipo: {tipo}")
    print(f"   ❓ {row[col_question]}")
    print(f"   ✅ {str(row[col_truth])[:200]}")

# Salva
output_path = project_root / 'outputs' / 'golden_dataset.csv'
output_path.parent.mkdir(exist_ok=True)
df_golden.to_csv(output_path, index=False)
print(f"\n💾 Salvo em: {output_path}")

---
## Parte 3 — Avaliação com Métricas RAGAS

Agora rodamos o RAG em cada pergunta do golden dataset e medimos:

| Métrica | Compara | Mede |
|---|---|---|
| **Faithfulness** | Answer ↔ Context | O LLM usou o contexto? |
| **Answer Correctness** | Answer ↔ Ground Truth | A resposta está certa? |
| **Answer Relevance** | Answer ↔ Question | A resposta é relevante? |
| **Context Precision** | Context ↔ Question+GT | O contexto é preciso? |
| **Context Recall** | Context ↔ Ground Truth | O contexto é completo? |

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import (
    faithfulness,
    answer_correctness,
    answer_relevancy,
    context_precision,
    context_recall,
)

# Roda o RAG em cada pergunta do golden dataset
ragas_data = {'question': [], 'answer': [], 'contexts': [], 'ground_truth': []}

for i, row in df_golden.iterrows():
    question = row.get('user_input', row.get('question', ''))
    ground_truth = row.get('reference', row.get('ground_truth', ''))
    result = rag.query(question)
    ragas_data['question'].append(question)
    ragas_data['answer'].append(result['answer'])
    ragas_data['contexts'].append(result['contexts'])
    ragas_data['ground_truth'].append(ground_truth)
    print(f'  [{i+1}/{len(df_golden)}] ✓')

print('\n✅ RAG executado em todas as perguntas!')

In [ ]:
# Avalia com RAGAS
dataset = Dataset.from_dict(ragas_data)

result = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_correctness, answer_relevancy, context_precision, context_recall]
)

df_scores = result.to_pandas()

# Scores médios
metric_cols = ['faithfulness', 'answer_correctness', 'answer_relevancy', 'context_precision', 'context_recall']
print('\n📊 SCORES MÉDIOS:')
print(df_scores[metric_cols].mean().round(3).to_string())

In [ ]:
# Visualiza scores por pergunta
df_scores[['question', 'faithfulness', 'answer_correctness', 'answer_relevancy']].round(3)

In [ ]:
# 🔍 Insight importante:
# Faithfulness baixa + Answer Correctness alta =
#   LLM usando conhecimento próprio, não o contexto recuperado!

problematic = df_scores[
    (df_scores['faithfulness'] < 0.5) & 
    (df_scores['answer_correctness'] > 0.6)
]

if len(problematic) > 0:
    print('⚠️  Perguntas onde o LLM usou conhecimento próprio (não o contexto):')
    for _, row in problematic.iterrows():
        print(f"   → {row['question']}")
        print(f"     faithfulness={row['faithfulness']:.2f}, correctness={row['answer_correctness']:.2f}")
else:
    print('✅ Todas as respostas corretas foram baseadas no contexto recuperado!')

# Salva resultados finais
scores_path = project_root / 'outputs' / 'evaluation_scores.csv'
df_scores.to_csv(scores_path, index=False)
print(f'\n💾 Scores salvos em: {scores_path}')